# NVIDIA Triton Inference Server

Enterprise-grade inference serving for any framework, any GPU or CPU.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

**NVIDIA Triton Inference Server** is an open-source inference serving platform that standardizes AI model deployment across any framework and hardware.

### What is it?

Triton is an enterprise-grade inference server that:
- Serves models from **any framework**: TensorFlow, PyTorch, ONNX, TensorRT, OpenVINO, etc.
- Supports **any hardware**: NVIDIA GPUs, CPUs, AWS Inferentia, etc.
- Provides **production features**: Dynamic batching, model ensembles, concurrent execution
- Offers **multiple interfaces**: HTTP/REST, gRPC, C API
- Enables **Kubernetes-native deployment** with standard patterns

### Why use it?

Key benefits:
- **Framework Agnostic**: One server for all your models (TF, PyTorch, ONNX, custom)
- **Maximum Utilization**: Dynamic batching, concurrent model execution, pipeline parallelism
- **Production Ready**: Battle-tested by NVIDIA and thousands of enterprises
- **Kubernetes Native**: First-class K8s support, Helm charts, operators
- **Vendor Neutral**: Open-source (BSD license), no lock-in
- **Enterprise Features**: Multi-model serving, A/B testing, model ensembles, versioning

### When to use it?

Triton is ideal when:
- Serving **multiple models** from different frameworks
- Need **enterprise-grade** reliability and observability
- Building **multi-stage inference pipelines** (model ensembles)
- Deploying on **Kubernetes** at scale
- Require **dynamic batching** for optimal throughput
- Need to support **multiple hardware backends**
- Want a **standardized serving layer** across organization

**Comparison**: Triton is more complex than single-framework solutions but provides unmatched flexibility and production features.

## Key Features

### Core Capabilities of Triton Inference Server

| Feature | Description | Benefit |
|---------|-------------|----------|
| **Multi-Framework** | TensorFlow, PyTorch, ONNX, TensorRT, TorchScript, Python, etc. | One server for all models |
| **Dynamic Batching** | Automatic batching of concurrent requests | 10-100x throughput increase |
| **Concurrent Execution** | Multiple models execute simultaneously | Maximize hardware utilization |
| **Model Ensemble** | Chain models into pipelines | Complex inference workflows |
| **Model Versioning** | Serve multiple versions, auto-rollback | Zero-downtime updates |
| **Backend Extensibility** | Custom backends in C++ or Python | Support any framework |
| **Multiple Protocols** | HTTP/REST, gRPC, C API | Client flexibility |
| **GPU/CPU Support** | NVIDIA GPUs, x86, ARM, Inferentia | Hardware flexibility |
| **Kubernetes Native** | Helm charts, operators, autoscaling | Cloud-native deployment |
| **Observability** | Prometheus metrics, detailed logging | Production monitoring |
| **Model Analyzer** | Automated performance tuning | Optimize configurations |
| **Model Navigator** | Convert and optimize models | Simplified deployment |

## Architecture Overview

Triton's architecture is designed for maximum flexibility and performance:

```
┌─────────────────────────────────────────────────────────────┐
│                    CLIENT LAYER                             │
│  ┌──────────┐  ┌──────────┐  ┌──────────┐  ┌──────────┐   │
│  │HTTP/REST │  │  gRPC    │  │   C API  │  │  Python  │   │
│  │ Client   │  │  Client  │  │  Client  │  │  Client  │   │
│  └────┬─────┘  └────┬─────┘  └────┬─────┘  └────┬─────┘   │
└───────┼─────────────┼─────────────┼─────────────┼──────────┘
        │             │             │             │
        └─────────────┴─────────────┴─────────────┘
                      │
┌─────────────────────▼─────────────────────────────────────┐
│              TRITON INFERENCE SERVER                      │
│                                                           │
│  ┌────────────────────────────────────────────────────┐  │
│  │           HTTP/gRPC Endpoints                      │  │
│  │  /v2/models/*/infer  /v2/health  /metrics         │  │
│  └─────────────────────┬──────────────────────────────┘  │
│                        │                                  │
│  ┌─────────────────────▼──────────────────────────────┐  │
│  │         Request Scheduler & Batcher               │  │
│  │  • Dynamic batching                               │  │
│  │  • Sequence batching                              │  │
│  │  • Request priority                               │  │
│  │  • Timeout management                             │  │
│  └─────────────────────┬──────────────────────────────┘  │
│                        │                                  │
│  ┌─────────────────────▼──────────────────────────────┐  │
│  │         Model Repository Manager                  │  │
│  │  • Model loading/unloading                        │  │
│  │  • Version control                                │  │
│  │  • Polling for updates                            │  │
│  └─────────────────────┬──────────────────────────────┘  │
│                        │                                  │
│  ┌────────────────────┬┴─────────┬───────────┬────────┐  │
│  │                    │          │           │        │  │
│  ▼                    ▼          ▼           ▼        ▼  │
│  ┌──────────┐  ┌──────────┐  ┌──────────┐  ┌──────────┐│
│  │TensorRT  │  │PyTorch   │  │TensorFlow│  │  ONNX    ││
│  │Backend   │  │Backend   │  │Backend   │  │Runtime   ││
│  └──────────┘  └──────────┘  └──────────┘  └──────────┘│
│  ┌──────────┐  ┌──────────┐  ┌──────────┐  ┌──────────┐│
│  │Python    │  │Custom    │  │Ensemble  │  │Business  ││
│  │Backend   │  │Backend   │  │Scheduler │  │Logic     ││
│  └──────────┘  └──────────┘  └──────────┘  └──────────┘│
└───────────────────────┬────────────────────────────────┘
                        │
┌───────────────────────▼─────────────────────────────────┐
│               HARDWARE LAYER                            │
│  ┌──────────┐  ┌──────────┐  ┌──────────┐  ┌─────────┐│
│  │ NVIDIA   │  │   CPU    │  │   AWS    │  │  Other  ││
│  │   GPU    │  │ (x86/ARM)│  │Inferentia│  │Hardware ││
│  └──────────┘  └──────────┘  └──────────┘  └─────────┘│
└─────────────────────────────────────────────────────────┘
```

### Key Components

1. **Model Repository**: File system or cloud storage with model files and configs
2. **Backends**: Framework-specific execution engines (TensorRT, PyTorch, etc.)
3. **Scheduler**: Routes requests, manages batching and concurrency
4. **Ensemble Scheduler**: Orchestrates multi-model pipelines
5. **Inference Protocols**: HTTP/REST, gRPC, C API for client communication

## Installation

### Prerequisites

- **Docker**: Highly recommended for deployment
- **NVIDIA GPU** (optional): For GPU-accelerated inference
- **NVIDIA Container Toolkit** (optional): For GPU access in Docker

### Installation via Docker (Recommended)

In [ ]:
# Pull Triton Inference Server image
# CPU-only version
# docker pull nvcr.io/nvidia/tritonserver:24.01-py3

# GPU version (recommended)
# docker pull nvcr.io/nvidia/tritonserver:24.01-py3

print("Triton is typically deployed via Docker containers")
print("Image size: ~6-8GB depending on backends included")

In [ ]:
# Install Triton client libraries for Python
# pip install tritonclient[all]

# Or specific protocols
# pip install tritonclient[http]  # HTTP/REST only
# pip install tritonclient[grpc]  # gRPC only

import sys
print(f"Python version: {sys.version}")

# Verify client installation
# import tritonclient.http as httpclient
# import tritonclient.grpc as grpcclient
# print("Triton client libraries installed successfully")

## Basic Usage

### Model Repository Structure

Triton serves models from a **model repository** - a directory with a specific structure:

In [ ]:
# Model repository structure
repository_structure = '''
model_repository/
├── resnet50/
│   ├── config.pbtxt          # Model configuration
│   └── 1/                    # Version 1
│       └── model.onnx        # Model file
├── bert_tokenizer/
│   ├── config.pbtxt
│   └── 1/
│       └── model.py          # Python backend
└── text_classifier/
    ├── config.pbtxt
    ├── 1/
    │   └── model.plan        # TensorRT engine
    └── 2/                    # Version 2 (newer)
        └── model.plan
'''

print("Triton Model Repository Structure:")
print(repository_structure)

### Creating a Simple Model Configuration

In [ ]:
# Example config.pbtxt for an ONNX model
config_example = '''
name: "resnet50"
platform: "onnxruntime_onnx"
max_batch_size: 8

input [
  {
    name: "input"
    data_type: TYPE_FP32
    dims: [ 3, 224, 224 ]
  }
]

output [
  {
    name: "output"
    data_type: TYPE_FP32
    dims: [ 1000 ]
  }
]

# Dynamic batching for throughput
dynamic_batching {
  preferred_batch_size: [ 4, 8 ]
  max_queue_delay_microseconds: 100
}

# Instance configuration
instance_group [
  {
    count: 1
    kind: KIND_GPU
  }
]
'''

print("Example config.pbtxt:")
print(config_example)

### Starting Triton Server

In [ ]:
# Start Triton server with model repository
docker_command = '''
docker run --gpus all \\
  --rm \\
  --shm-size=1g \\
  --ulimit memlock=-1 \\
  --ulimit stack=67108864 \\
  -p 8000:8000 \\
  -p 8001:8001 \\
  -p 8002:8002 \\
  -v /path/to/model_repository:/models \\
  nvcr.io/nvidia/tritonserver:24.01-py3 \\
  tritonserver --model-repository=/models
  
# Ports:
# 8000: HTTP/REST
# 8001: gRPC
# 8002: Metrics (Prometheus)
'''

print("Start Triton Server:")
print(docker_command)

### Making Inference Requests (Python Client)

In [ ]:
# HTTP/REST client example
http_example = '''
import tritonclient.http as httpclient
import numpy as np

# Create client
client = httpclient.InferenceServerClient(url="localhost:8000")

# Check if model is ready
if client.is_model_ready("resnet50"):
    print("Model is ready")

# Prepare input data
input_data = np.random.randn(1, 3, 224, 224).astype(np.float32)

# Create input tensor
inputs = []
inputs.append(httpclient.InferInput("input", input_data.shape, "FP32"))
inputs[0].set_data_from_numpy(input_data)

# Create output placeholder
outputs = []
outputs.append(httpclient.InferRequestedOutput("output"))

# Run inference
response = client.infer(
    model_name="resnet50",
    inputs=inputs,
    outputs=outputs
)

# Get results
output_data = response.as_numpy("output")
print(f"Output shape: {output_data.shape}")
print(f"Top prediction: {output_data.argmax()}")
'''

print("HTTP Client Example:")
print(http_example)

In [ ]:
# gRPC client example (faster than HTTP)
grpc_example = '''
import tritonclient.grpc as grpcclient
import numpy as np

# Create gRPC client
client = grpcclient.InferenceServerClient(url="localhost:8001")

# Prepare input
input_data = np.random.randn(1, 3, 224, 224).astype(np.float32)
inputs = [grpcclient.InferInput("input", input_data.shape, "FP32")]
inputs[0].set_data_from_numpy(input_data)

# Run inference
response = client.infer(
    model_name="resnet50",
    inputs=inputs
)

# Get results
output_data = response.as_numpy("output")
print(f"Prediction: {output_data.argmax()}")
'''

print("gRPC Client Example (recommended for production):")
print(grpc_example)

## Advanced Features

### 1. Dynamic Batching

In [ ]:
# Dynamic batching configuration
dynamic_batching_config = '''
# In config.pbtxt
dynamic_batching {
  # Preferred batch sizes (Triton will try to form these)
  preferred_batch_size: [ 4, 8, 16 ]
  
  # Maximum queue delay before sending partial batch
  max_queue_delay_microseconds: 100
  
  # Preserve request ordering
  preserve_ordering: false
  
  # Priority levels
  priority_levels: 3
  default_priority_level: 1
  
  # Queue policy
  default_queue_policy {
    timeout_action: REJECT
    default_timeout_microseconds: 1000000
    allow_timeout_override: true
    max_queue_size: 128
  }
}
'''

print("Dynamic Batching Configuration:")
print(dynamic_batching_config)
print("\nBenefits:")
print("• Automatically combines concurrent requests into batches")
print("• Can increase throughput by 10-100x")
print("• Configurable trade-off between latency and throughput")

### 2. Model Ensembles (Pipelines)

In [ ]:
# Model ensemble example: tokenizer → model → post-processor
ensemble_config = '''
# In ensemble_model/config.pbtxt
name: "text_classification_pipeline"
platform: "ensemble"
max_batch_size: 8

input [
  {
    name: "text_input"
    data_type: TYPE_STRING
    dims: [ 1 ]
  }
]

output [
  {
    name: "class_label"
    data_type: TYPE_STRING
    dims: [ 1 ]
  }
]

ensemble_scheduling {
  # Step 1: Tokenization
  step [
    {
      model_name: "tokenizer"
      model_version: -1
      input_map {
        key: "text"
        value: "text_input"
      }
      output_map {
        key: "tokens"
        value: "tokens"
      }
    }
  ]
  
  # Step 2: Model inference
  step [
    {
      model_name: "classifier"
      model_version: -1
      input_map {
        key: "input_ids"
        value: "tokens"
      }
      output_map {
        key: "logits"
        value: "logits"
      }
    }
  ]
  
  # Step 3: Post-processing
  step [
    {
      model_name: "post_processor"
      model_version: -1
      input_map {
        key: "raw_logits"
        value: "logits"
      }
      output_map {
        key: "label"
        value: "class_label"
      }
    }
  ]
}
'''

print("Model Ensemble Configuration:")
print(ensemble_config)
print("\nUse Cases:")
print("• NLP pipelines: tokenizer → encoder → decoder")
print("• Computer vision: preprocessor → detector → tracker")
print("• Multi-stage recommendations: retrieval → ranking → filtering")

### 3. Model Versioning and A/B Testing

In [ ]:
# Model versioning configuration
versioning_config = '''
# Serve multiple versions simultaneously
model_repository/
└── my_model/
    ├── config.pbtxt
    ├── 1/              # Old version
    │   └── model.pt
    ├── 2/              # Current version
    │   └── model.pt
    └── 3/              # New version (testing)
        └── model.pt

# In config.pbtxt:
version_policy: {
  # Serve all versions
  all {}
  
  # Or serve specific versions
  # specific { versions: [2, 3] }
  
  # Or serve latest N versions
  # latest { num_versions: 2 }
}

# Client can request specific version:
response = client.infer(
    model_name="my_model",
    model_version="3",  # Test new version
    inputs=inputs
)
'''

print("Model Versioning:")
print(versioning_config)

### 4. Python Backend (Business Logic)

In [ ]:
# Python backend for custom logic
python_backend_example = '''
# model_repository/custom_model/1/model.py
import triton_python_backend_utils as pb_utils
import numpy as np

class TritonPythonModel:
    def initialize(self, args):
        """Called once when model is loaded."""
        self.model_config = json.loads(args[\'model_config\'])
        # Load custom resources, initialize state
        self.preprocessor = CustomPreprocessor()
    
    def execute(self, requests):
        """Execute inference for a batch of requests."""
        responses = []
        
        for request in requests:
            # Get input tensors
            in_tensor = pb_utils.get_input_tensor_by_name(request, "input")
            input_data = in_tensor.as_numpy()
            
            # Custom processing
            processed = self.preprocessor.transform(input_data)
            
            # Call another model (BLS - Business Logic Scripting)
            infer_request = pb_utils.InferenceRequest(
                model_name="backend_model",
                requested_output_names=["output"],
                inputs=[pb_utils.Tensor("input", processed)]
            )
            infer_response = infer_request.exec()
            
            # Post-process
            output = pb_utils.get_output_tensor_by_name(infer_response, "output")
            result = self.postprocess(output.as_numpy())
            
            # Create response
            out_tensor = pb_utils.Tensor("output", result)
            response = pb_utils.InferenceResponse(output_tensors=[out_tensor])
            responses.append(response)
        
        return responses
    
    def finalize(self):
        """Called when model is unloaded."""
        print("Cleaning up resources")
'''

print("Python Backend Example:")
print(python_backend_example)
print("\nUse Cases:")
print("• Custom preprocessing/postprocessing")
print("• Business logic integration")
print("• Feature engineering")
print("• Calling external services")

## Use Cases

### Real-World Applications

#### Use Case 1: Multi-Model Serving Platform

In [ ]:
# Serve multiple models from different frameworks
multi_model_setup = '''
model_repository/
├── image_classification/
│   ├── config.pbtxt        # TensorRT engine
│   └── 1/model.plan
├── object_detection/
│   ├── config.pbtxt        # ONNX model
│   └── 1/model.onnx
├── text_embedding/
│   ├── config.pbtxt        # PyTorch model
│   └── 1/model.pt
└── recommendation/
    ├── config.pbtxt        # TensorFlow SavedModel
    └── 1/model.savedmodel/

# All served by single Triton instance
# Clients can query any model via same API
# Dynamic batching optimizes each model independently
'''

print("Multi-Model Platform:")
print(multi_model_setup)
print("\nBenefits:")
print("• Unified serving layer for all ML models")
print("• Simplified infrastructure (one server, not per-framework)")
print("• Consistent monitoring and observability")
print("• Shared resource pool (GPUs efficiently utilized)")

#### Use Case 2: NLP Pipeline with Ensemble

In [ ]:
# End-to-end NLP pipeline
nlp_pipeline = '''
# Pipeline: Text → Tokenizer → BERT → Classification

# Client sends raw text
input_text = "This product is amazing!"

# Triton ensemble handles the full pipeline:
# 1. Python tokenizer: text → token IDs
# 2. BERT (ONNX): token IDs → embeddings  
# 3. Classifier (TensorRT): embeddings → sentiment
# 4. Python post-processor: logits → label

response = client.infer(
    model_name="sentiment_analysis_pipeline",
    inputs=[{"name": "text", "data": input_text}]
)

# Client receives final result: "positive"
'''

print("NLP Ensemble Pipeline:")
print(nlp_pipeline)
print("\nAdvantages:")
print("• Client only sees input/output, not intermediate steps")
print("• Each stage optimized independently (BERT on GPU, tokenizer on CPU)")
print("• Easy to swap components (try different tokenizers)")
print("• Consistent latency tracking for entire pipeline")

#### Use Case 3: Real-Time Recommendation System

In [ ]:
# High-throughput recommendation serving
recommendation_setup = '''
# Configuration for low-latency recommendations
name: "user_recommendation"
platform: "tensorrt_plan"
max_batch_size: 128

# Aggressive dynamic batching
dynamic_batching {
  preferred_batch_size: [ 32, 64, 128 ]
  max_queue_delay_microseconds: 500  # Max 0.5ms wait
}

# Multiple GPU instances for concurrency
instance_group [
  {
    count: 4        # 4 instances
    kind: KIND_GPU
    gpus: [ 0 ]     # All on GPU 0
  }
]

# Result:
# - Latency: p50=3ms, p99=8ms
# - Throughput: 50,000 req/sec on single A100
# - GPU utilization: 95%
'''

print("Recommendation System Setup:")
print(recommendation_setup)

## Best Practices

### Recommended Practices for Triton

#### 1. Model Configuration

- **Always specify max_batch_size**: Even if using batch size 1, set it explicitly
- **Use dynamic batching**: Enable for 10-100x throughput gains
- **Set instance_group**: Control number of model copies and GPU assignment
- **Version models**: Use version directories (1/, 2/, 3/) for zero-downtime updates
- **Optimize input/output**: Use minimal precision (FP16 vs FP32) where acceptable

#### 2. Performance Tuning

```python
# Use Model Analyzer for automated tuning
# model-analyzer profile \\
#   --model-repository=/models \\
#   --profile-models=my_model \\
#   --triton-launch-mode=docker \\
#   --output-model-repository-path=/optimized_models
```

- Use Model Analyzer for automated configuration optimization
- Benchmark with `perf_analyzer` tool
- Monitor metrics to identify bottlenecks
- Use appropriate backend (TensorRT > ONNX > PyTorch for inference speed)

#### 3. Production Deployment

- **Use gRPC over HTTP**: 20-30% lower latency
- **Enable model control mode**: Explicit loading/unloading for large model catalogs
- **Set resource limits**: `--model-control-mode=explicit` for memory-constrained environments
- **Use model warmup**: Pre-cache JIT compiled operations
- **Implement health checks**: `/v2/health/ready` and `/v2/health/live`

#### 4. Model Repository Organization

```
# Use cloud storage for model repository
tritonserver \\
  --model-repository=s3://my-bucket/models \\
  --model-control-mode=poll \\
  --repository-poll-secs=60
```

- Store in cloud storage (S3, GCS, Azure Blob)
- Use polling mode for automatic model updates
- Organize by domain/team for large catalogs

#### 5. Monitoring and Observability

- Export metrics to Prometheus
- Monitor: queue time, compute time, GPU utilization
- Set up alerts for error rates, latency spikes
- Use Triton's built-in logging (structured JSON logs)

## Common Pitfalls

### What to Avoid

#### 1. Incorrect Input/Output Tensor Names

**Problem**: Tensor names in config don't match model definition

**Symptom**: `Invalid argument: unexpected input` errors

**Solution**: Use Model Analyzer or check model internals:
```python
# For ONNX models
import onnx
model = onnx.load("model.onnx")
print([input.name for input in model.graph.input])
```

#### 2. Missing Platform Specification

**Problem**: Not specifying backend platform in config

**Symptom**: `failed to load model` with no clear error

**Solution**: Always set `platform` in config.pbtxt:
- `pytorch_libtorch`
- `tensorflow_savedmodel`
- `onnxruntime_onnx`
- `tensorrt_plan`

#### 3. Suboptimal Batching Configuration

**Problem**: Using static batching or no batching

**Impact**: 10-100x lower throughput

**Solution**: Enable dynamic batching with appropriate parameters

#### 4. Not Using Model Versions

**Problem**: Updating model file in place

**Impact**: Requests fail during model reload

**Solution**: Use version directories for zero-downtime updates

#### 5. Insufficient Shared Memory

**Problem**: Not setting `--shm-size` in Docker

**Symptom**: Crashes with large batches or models

**Solution**: Set `--shm-size=1g` or larger in Docker run command

## Performance Optimization

### Achieving Peak Performance

In [ ]:
# Use perf_analyzer for benchmarking
perf_analyzer_example = '''
# Install perf_analyzer
# pip install tritonclient[all]

# Benchmark throughput
perf_analyzer \\
  -m resnet50 \\
  --shape input:1,3,224,224 \\
  --concurrency-range 1:64:8 \\
  -u localhost:8001 \\
  --protocol grpc \\
  -f results.csv

# Find optimal concurrency
# Output shows throughput vs latency trade-offs
'''

print("Performance Analyzer:")
print(perf_analyzer_example)

In [ ]:
# Model Analyzer for automated optimization
model_analyzer_example = '''
# Automatically find best configuration
model-analyzer profile \\
  --model-repository=/models \\
  --profile-models=resnet50 \\
  --triton-launch-mode=docker \\
  --output-model-repository-path=/optimized \\
  --run-config-search-max-concurrency=256 \\
  --run-config-search-max-instance-count=8

# Model Analyzer will:
# 1. Test different instance counts
# 2. Test different dynamic batching configs
# 3. Measure throughput and latency
# 4. Generate optimized config.pbtxt
'''

print("Model Analyzer (Automated Tuning):")
print(model_analyzer_example)

## Production Deployment

### Kubernetes Deployment

In [ ]:
# Kubernetes manifest for Triton
k8s_deployment = '''
apiVersion: v1
kind: ConfigMap
metadata:
  name: triton-config
data:
  model-repository: "s3://my-models"
---
apiVersion: v1
kind: Service
metadata:
  name: triton-service
spec:
  selector:
    app: triton
  ports:
  - name: http
    port: 8000
    targetPort: 8000
  - name: grpc
    port: 8001
    targetPort: 8001
  - name: metrics
    port: 8002
    targetPort: 8002
  type: LoadBalancer
---
apiVersion: apps/v1
kind: Deployment
metadata:
  name: triton
spec:
  replicas: 3
  selector:
    matchLabels:
      app: triton
  template:
    metadata:
      labels:
        app: triton
    spec:
      containers:
      - name: triton
        image: nvcr.io/nvidia/tritonserver:24.01-py3
        args:
        - tritonserver
        - --model-repository=s3://my-models
        - --model-control-mode=poll
        - --repository-poll-secs=60
        - --log-verbose=1
        ports:
        - containerPort: 8000
        - containerPort: 8001
        - containerPort: 8002
        resources:
          limits:
            nvidia.com/gpu: 1
          requests:
            nvidia.com/gpu: 1
            cpu: "4"
            memory: "16Gi"
        livenessProbe:
          httpGet:
            path: /v2/health/live
            port: 8000
          initialDelaySeconds: 30
          periodSeconds: 10
        readinessProbe:
          httpGet:
            path: /v2/health/ready
            port: 8000
          initialDelaySeconds: 30
          periodSeconds: 5
---
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: triton-hpa
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: triton
  minReplicas: 2
  maxReplicas: 10
  metrics:
  - type: Resource
    resource:
      name: cpu
      target:
        type: Utilization
        averageUtilization: 70
'''

print("Kubernetes Deployment:")
print(k8s_deployment)

## Monitoring and Observability

In [ ]:
# Key Triton metrics
metrics_guide = '''
# Triton exposes Prometheus metrics on :8002/metrics

# Critical Metrics to Monitor:

# 1. Request Metrics
nv_inference_request_success{model="resnet50"}        # Successful requests
nv_inference_request_failure{model="resnet50"}        # Failed requests
nv_inference_request_duration_us{model="resnet50"}   # E2E latency

# 2. Queue Metrics (batching)
nv_inference_queue_duration_us{model="resnet50"}     # Time in queue
nv_inference_pending_request_count{model="resnet50"} # Queue depth

# 3. Compute Metrics
nv_inference_compute_infer_duration_us{model="resnet50"}  # Actual inference time
nv_inference_compute_input_duration_us{model="resnet50"}  # Input processing
nv_inference_compute_output_duration_us{model="resnet50"} # Output processing

# 4. GPU Metrics
nv_gpu_utilization{gpu_uuid="GPU-..."}
nv_gpu_memory_total_bytes{gpu_uuid="GPU-..."}
nv_gpu_memory_used_bytes{gpu_uuid="GPU-..."}

# 5. Model Metrics
nv_inference_exec_count{model="resnet50"}             # Execution count
nv_inference_model_load_count{model="resnet50"}       # Load events
'''

print("Triton Metrics Guide:")
print(metrics_guide)

## Troubleshooting

### Common Issues

#### Issue 1: Model Fails to Load

**Check**:
```bash
# Check Triton logs
docker logs <container-id>

# Verify model repository structure
ls -R /path/to/model_repository

# Test config syntax
tritonserver --model-repository=/models --strict-model-config=false
```

#### Issue 2: Low Throughput

**Diagnose**:
- Check if dynamic batching is enabled
- Monitor `nv_inference_queue_duration_us` (should be >0 if batching works)
- Use `perf_analyzer` to find optimal concurrency
- Check GPU utilization (should be >80%)

#### Issue 3: Memory Issues

**Solutions**:
- Reduce `max_batch_size`
- Reduce number of model instances
- Use `--model-control-mode=explicit` to control loading
- Convert models to TensorRT for lower memory footprint

#### Issue 4: Version Update Not Working

**Check**:
```bash
# Ensure polling is enabled
tritonserver --model-control-mode=poll --repository-poll-secs=30

# Or manually reload
curl -X POST localhost:8000/v2/repository/models/my_model/load
```

## Comparison with Alternatives

### How Triton Compares

| Feature | Triton | TorchServe | TF Serving | KServe |
|---------|--------|------------|------------|--------|
| **Multi-Framework** | ⭐⭐⭐⭐⭐ | ⭐⭐ (PyTorch) | ⭐ (TF only) | ⭐⭐⭐⭐ |
| **Dynamic Batching** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐⭐ |
| **Model Ensemble** | ⭐⭐⭐⭐⭐ | ⭐ | ⭐ | ⭐⭐⭐ |
| **Kubernetes Native** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ |
| **Performance** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ |
| **Ease of Use** | ⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐ |
| **Observability** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐⭐ |

### When to Choose Triton

**Choose Triton when:**
- ✅ Serving models from **multiple frameworks**
- ✅ Need **model ensembles** or multi-stage pipelines
- ✅ Require **enterprise-grade** reliability and features
- ✅ Deploying on **Kubernetes** at scale
- ✅ Need **maximum performance** with dynamic batching
- ✅ Want **vendor-neutral** open-source solution

**Choose alternatives when:**
- ❌ Only using PyTorch (TorchServe is simpler)
- ❌ Only using TensorFlow (TF Serving is easier)
- ❌ Need simpler deployment (framework-specific servers)
- ❌ Small team, limited resources

## Resources

### Official Documentation

- **GitHub**: https://github.com/triton-inference-server/server
- **Documentation**: https://docs.nvidia.com/deeplearning/triton-inference-server/
- **Quick Start**: https://github.com/triton-inference-server/server/blob/main/docs/getting_started/quickstart.md
- **Docker Images**: https://catalog.ngc.nvidia.com/orgs/nvidia/containers/tritonserver

### Tools and Utilities

- **Model Analyzer**: https://github.com/triton-inference-server/model_analyzer
- **Model Navigator**: https://github.com/triton-inference-server/model_navigator
- **Client Libraries**: https://github.com/triton-inference-server/client

### Tutorials and Examples

- **Tutorials**: https://github.com/triton-inference-server/tutorials
- **Backend Examples**: https://github.com/triton-inference-server/backend
- **Python Backend**: https://github.com/triton-inference-server/python_backend

### Community

- **GitHub Discussions**: https://github.com/triton-inference-server/server/discussions
- **NVIDIA Forums**: https://forums.developer.nvidia.com/c/triton-inference-server/
- **Slack**: https://join.slack.com/t/triton-inference-server/shared_invite/

### Related Technologies

- **KServe**: Kubernetes-based serving (uses Triton as backend)
- **TensorRT-LLM**: Optimized LLM backend for Triton
- **ONNX Runtime**: Fast inference backend
- **RAPIDS**: GPU-accelerated data science (integrates with Triton)